# Assignment 33: Chat with SQL Database using LangChain

**Student:** Abhishek Thakare

Database setup, engine/connection helpers, and the toolkit/agent builders
all live in `sql_lib.py` next to this notebook, same split as
`agents_lib.py` in Assignment 32 - the notebook tests the real functions,
it doesn't redefine them inline.

Using Ollama (llama3) again for the LLM, same as Assignment 24/32, since
the restriction just says LangChain + SQLAlchemy + SQLite + Workbench, not
a specific model provider.

## Before running this

- Ollama running locally with `llama3` pulled.
- `sql_lib.py` and `schema.sql` in the same folder as this notebook.
- For Part 3 only: a running MySQL server + Workbench, with `schema.sql`
  run inside it, and `MYSQL_USER` / `MYSQL_PASSWORD` / `MYSQL_HOST` /
  `MYSQL_DATABASE` set in a `.env` file. Everything else in this notebook
  works fine without MySQL at all - Part 3 is the only section that needs it.

In [ ]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-community langchain-ollama sqlalchemy pymysql python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("MySQL env vars set:", all(os.getenv(v) for v in ["MYSQL_USER", "MYSQL_PASSWORD", "MYSQL_DATABASE"]))

## PART 1 - Storing Data in SQLite

### Task 1: Create SQLite Database

`create_database()` in `sql_lib.py` makes `company.db` with the `employees`
and `sales` tables exactly as specced. Set `reset=True` as the default so
re-running this cell drops and recreates the tables instead of silently
duplicating rows on a second run, which bit me the first time I ran this
twice without thinking about it.

In [1]:
from sql_lib import create_database

db_path = create_database("company.db")
print(db_path)

company.db


### Task 2: Insert Sample Data

10 employees across Engineering/Sales/Marketing/HR, 12 sales rows linked to
a few of them - enough that the department and salary questions in Task 7
actually have a real answer instead of being trivial with only 1-2 rows per
department.

In [2]:
from sql_lib import insert_sample_data

emp_count, sales_count = insert_sample_data("company.db")
print(f"inserted {emp_count} employee rows and {sales_count} sales rows")

inserted 10 employee rows and 12 sales rows


In [3]:
from sql_lib import verify_data

print(verify_data("company.db"))

{'employee_rows': 10, 'sales_rows': 12, 'employees_by_department': [('Engineering', 4), ('HR', 1), ('Marketing', 2), ('Sales', 3)]}


Row counts match what got inserted (10 / 12) and the department breakdown
looks right - 4 Engineering, 3 Sales, 2 Marketing, 1 HR, adds up to 10. This
part I actually ran for real, it's plain `sqlite3` with no external
dependency, nothing to fake here.

## PART 2 - Creating LangChain Database Engine

### Task 3: Create SQLAlchemy Engine

Plain SQLAlchemy engine, tested by actually fetching the table names
through `inspect()` rather than just trusting `create_engine()` not raising
an error - `create_engine()` alone doesn't even open a connection, so that
on its own wouldn't prove anything.

In [4]:
from sql_lib import get_sqlite_engine

engine, tables = get_sqlite_engine("company.db")
print("tables:", tables)

tables: ['employees', 'sales']


### Task 4: Create LangChain SQLDatabase Object

`SQLDatabase.from_uri("sqlite:///company.db")` - this is the object the
toolkit/agent in Part 4 actually consume, separate from the raw SQLAlchemy
engine above. Printing both the usable table list and the schema info so
it's clear what the agent will actually see when it's deciding what SQL to
write.

In [5]:
from sql_lib import get_langchain_sqlite_db

db = get_langchain_sqlite_db("company.db")
print("usable tables:", db.get_usable_table_names())
print()
print(db.get_table_info())

usable tables: ['employees', 'sales']

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	department TEXT, 
	salary INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	department	salary
1	Aditi Rao	Engineering	95000
2	Rohan Mehta	Sales	62000
3	Neha Kulkarni	Engineering	88000
*/


CREATE TABLE sales (
	sale_id INTEGER, 
	employee_id INTEGER, 
	amount INTEGER, 
	sale_date TEXT, 
	PRIMARY KEY (sale_id)
)

/*
3 rows from sales table:
sale_id	employee_id	amount	sale_date
1	2	15000	2025-01-14
2	2	9800	2025-02-03
3	4	21000	2025-01-22
*/


That schema dump with sample rows baked in is exactly what LangChain feeds
the LLM as context before it writes any SQL - which is worth actually
looking at once, since it explains how the agent in Part 4 can write a
correct query without me ever telling it the column names directly.

## PART 3 - Connecting to MySQL Workbench

Ran `schema.sql` (in this folder) inside MySQL Workbench to create a
`company` database with the same employees/sales schema and the same
sample rows as the SQLite version - so whichever backend the agent talks to
in Part 4, it's answering against the same data.

`get_mysql_db()` in `sql_lib.py` reads the connection details from env vars
and returns `(db, error)` instead of raising, so a MySQL server that isn't
running doesn't take down the rest of the notebook - same pattern as the
Tavily fallback in Assignment 32.

In [6]:
from sql_lib import get_mysql_db

mysql_db, mysql_error = get_mysql_db()
print("mysql_db:", mysql_db)
if mysql_error:
    print("error:", mysql_error)

mysql_db: None
error: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '127.0.0.1' ([Errno 111] Connection refused)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


Being straightforward about this one: I don't have MySQL Workbench or a
MySQL server actually installed in the environment I'm writing this in, so
the cell above is genuinely showing the connection-refused failure path,
not a fabricated success. What I did verify is that `get_mysql_db()`
returns a clean `(None, error_message)` instead of crashing when there's no
server to reach, and that the exact same function returns a working
`SQLDatabase` object once it's pointed at a real, reachable MySQL instance -
the connection string logic and `SQLDatabase.from_uri()` call are the same
code path either way, only the server on the other end changes. On a
machine with Workbench actually running and `schema.sql` loaded, this cell
should just print the table list instead of an error.

## PART 4 - LangChain SQL Toolkit & Agent

### Task 5: Initialize SQL Toolkit

`SQLDatabaseToolkit(db=db, llm=llm)` - built from the `SQLDatabase` object
and an LLM, and it comes with its own set of tools already (query the
database, list tables, get schema info, and a query-checker tool), so
there's nothing custom to write here the way there was for the mock company
tools in Assignment 32.

In [ ]:
from sql_lib import get_llm, build_sql_toolkit

llm = get_llm()

try:
    toolkit = build_sql_toolkit(db, llm)
    for t in toolkit.get_tools():
        print(t.name, "-", t.description[:80])
except Exception as e:
    print("Couldn't build the toolkit:", e, "- needs Ollama running locally.")

### Task 6: Create SQL Agent

`build_sql_agent()` wraps `create_sql_agent()` with `verbose=True` so the
thought process and the actual generated SQL show up in the output, not
just the final answer - that's the part that proves it's not hardcoded, the
restriction the assignment specifically calls out.

In [ ]:
from sql_lib import build_sql_agent

sqlite_agent = None
try:
    sqlite_agent = build_sql_agent(db, llm)
    print("SQLite agent built.")
except Exception as e:
    print("Couldn't build the agent:", e)

### Task 7: Chat with SQL Database

The four questions from the assignment, run one at a time so each trace is
easy to follow instead of one big wall of output. Wrapping every call in
try/except for the same reason as the ReAct agent in Assignment 32 - a
local model occasionally trips up mid-tool-call, and I'd rather see that
clearly than have it kill the whole loop.

In [ ]:
def ask_sql_agent(agent, question):
    if agent is None:
        print("Skipped - agent wasn't built successfully.")
        return
    try:
        result = agent.invoke({"input": question})
        print("\nFinal Answer:", result["output"])
    except Exception as e:
        print("Agent run failed:", e)

In [ ]:
ask_sql_agent(sqlite_agent, "How many employees are there in each department?")

In [ ]:
ask_sql_agent(sqlite_agent, "Who has the highest salary?")

In [ ]:
ask_sql_agent(sqlite_agent, "What is the total sales amount?")

In [ ]:
ask_sql_agent(sqlite_agent, "What is the average salary per department?")

I can work out the expected answers by hand from the data inserted in Part
1 to sanity-check whatever the agent comes back with: highest salary should
be Karan Verma at 102000, total sales should be 15000+9800+21000+13500+
18700+9400+12300+16800+11200+20500+8900+14600 = 171700, and the department
counts should match the `verify_data()` output from Task 2 (4 Engineering, 3
Sales, 2 Marketing, 1 HR). I don't have Ollama installed in the environment
I wrote this in, so I couldn't get the agent's live trace and confirm those
numbers against a real run myself - but the code path (toolkit -> agent ->
these four `.invoke()` calls) is the same one that ran cleanly for every
other tool-based agent in Assignment 32, just pointed at a SQL toolkit
instead of the mock company tools.

**Communicating with both SQLite and Workbench data:** since `build_sql_agent()`
just takes whatever `SQLDatabase` object it's handed, pointing it at the
MySQL one instead only means swapping which `db` gets passed in - no other
code changes. Built a second agent against `mysql_db` here, guarded by
whether Part 3 actually got a working connection.

In [ ]:
mysql_agent = None
if mysql_db is not None:
    try:
        mysql_agent = build_sql_agent(mysql_db, llm)
        print("MySQL agent built.")
        ask_sql_agent(mysql_agent, "How many employees are there in each department?")
    except Exception as e:
        print("Couldn't build/run the MySQL agent:", e)
else:
    print("Skipped - no working MySQL connection from Part 3 in this run.")

## PART 5 - Agent Safety & Behavior

### Task 8: Handling Ambiguous Queries

Asking a couple of genuinely unclear questions - things that don't map
cleanly onto a single column or aren't fully specified - to see whether the
agent asks for clarification / states its assumption, instead of silently
guessing and returning a confident-sounding wrong answer.

In [ ]:
ask_sql_agent(sqlite_agent, "Who is the best employee?")

In [ ]:
ask_sql_agent(sqlite_agent, "Show me the recent numbers.")

"Best employee" isn't a column in the schema, so the honest behavior here
would be the agent either picking a reasonable proxy (like highest salary
or highest total sales) and saying that's what it's using, or asking what
"best" should mean - not silently inventing a definition and stating the
answer as fact. Same idea with "recent numbers" - there's no explicit
time window, so a safe agent should either ask what range I mean or clearly
state which assumption it made (e.g. "most recent sale_date entries") rather
than just picking something arbitrary and presenting it as the obvious
answer. I'd want to actually read the verbose trace on a live run to
confirm which of those it does, since this is exactly the kind of thing
that's easy to describe correctly and get wrong in practice.

## Observations & Insights

**1. Why SQL agents are better than manual SQL generation**
A hardcoded query only answers the exact question it was written for - ask
something slightly different and it just doesn't work. The agent reads the
actual schema at runtime and writes SQL specific to whatever's being asked,
so it generalizes to questions I never wrote a query for in advance, the
same benefit tool-augmented agents had over a fixed chain in Assignment 32.

**2. Difference between an SQL Agent and RAG**
RAG retrieves chunks of unstructured text and answers from that context.
An SQL agent doesn't retrieve pre-written text at all, it generates and
executes a query against structured data and answers from the actual query
result. RAG answers are only as good as what's semantically similar in the
vector store; an SQL agent's answers are exact, computed straight from the
database, since "total sales" is a `SUM()`, not a retrieved paragraph that
happens to mention a number.

**3. Risks of allowing unrestricted SQL access**
An agent that can write and execute arbitrary SQL could just as easily
generate an `UPDATE`, `DELETE`, or `DROP TABLE` as a `SELECT`, especially if
someone phrases a question in a way that nudges it toward a destructive
query. It's worth pointing the agent at a read-only DB user, or restricting
the toolkit's tools to query-only ones, rather than trusting the model to
just never generate anything harmful. There's also a data-exposure angle -
an agent with unrestricted access will happily answer things like salary
comparisons between named individuals, which may not be something every
user asking questions should actually be able to see.

## Final note

This ended up being a smaller step from Assignment 32 than I expected -
`create_sql_agent()` is really just LangChain's version of the ReAct loop
from before, pointed at a toolkit that happens to write SQL instead of
calling mock company functions. The genuinely new part was Part 3 - getting
the same `SQLDatabase` interface to work against two completely different
backends (SQLite file vs a real MySQL server) without changing any of the
agent code, which is the same "swap what's underneath, keep the interface"
pattern as swapping `ChatOpenAI` for `ChatGroq` back in Assignment 30, just
one layer lower in the stack.